# Notebook 17: Regularization Techniques for Transformer Fine-Tuning

## 📚 Historical Context

**Timeline**: 2012-present - The battle against overfitting

**The Overfitting Problem (Pre-2012)**:
- **Neural networks**: Memorize training data perfectly
- **Poor generalization**: High train accuracy, low test accuracy
- **Small datasets**: Made overfitting worse
- **Solution**: Early stopping, L2 regularization (limited effectiveness)

**Key Innovations**:

**2012: Dropout (Hinton et al.)**
- **ImageNet breakthrough**: Dropout in AlexNet
- **Idea**: Randomly drop neurons during training
- **Effect**: Prevents co-adaptation, acts like ensemble
- **Impact**: Became standard in all deep learning

**2014: Adam Optimizer (Kingma & Ba)**
- **Adaptive learning rates** per parameter
- **Problem**: Implicit L2 regularization was wrong
- **Led to**: AdamW (2017)

**2017: AdamW - Decoupled Weight Decay (Loshchilov & Hutter)**
- **Key insight**: Weight decay ≠ L2 regularization (for Adam)
- **Proper implementation**: Decouple weight decay from gradient
- **Impact**: Better generalization, now default for transformers

**2017: Transformers with Regularization**
- **"Attention is All You Need"**: Dropout + Label Smoothing
- **Dropout locations**: Attention, FFN, embeddings
- **Found**: 0.1 dropout works well

**2018: BERT Regularization Strategy**
- **Weight decay**: 0.01
- **Dropout**: 0.1 everywhere
- **Warmup**: 10% of training steps
- **Became**: Standard for all transformer fine-tuning

**2019-Present: Modern Practices**
- **Gradient clipping**: Prevents exploding gradients
- **Learning rate warmup**: Stabilizes early training
- **Early stopping**: With patience for transformers
- **Combinations**: All techniques used together

## 🎯 What You'll Learn

1. **Dropout in Transformers** - Where and why to apply
2. **Weight Decay** - AdamW vs Adam, why it matters
3. **Early Stopping** - Patience and strategies
4. **Gradient Clipping** - Preventing instability
5. **Learning Rate Warmup** - Smooth training starts
6. **Comprehensive Comparison** - Effect on validation loss

## 💡 Why This Matters

**Regularization is critical for fine-tuning**:
- **Small datasets**: Easy to overfit (common in fine-tuning)
- **Large models**: Millions/billions of parameters
- **Without regularization**: Perfect train accuracy, terrible test accuracy
- **With regularization**: Balanced performance, better generalization

**Real-world impact**:
```
Without proper regularization:
- Train accuracy: 99%
- Test accuracy:  65%
- Model: Unusable

With proper regularization:
- Train accuracy: 92%
- Test accuracy:  88%
- Model: Production ready
```

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import copy

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Dropout in Transformers

### What is Dropout?

**Core idea**: During training, randomly set neurons to 0 with probability `p`

```python
# Training: Randomly drop
mask = (torch.rand(x.shape) > dropout_p).float()
x = x * mask / (1 - dropout_p)  # Scale to maintain expected value

# Inference: Use all neurons
x = x  # No dropout
```

### Why Dropout Works:

**1. Prevents Co-adaptation**
```
Without dropout: Neuron A relies on specific pattern from Neuron B
With dropout:    Neuron A must work even when B is dropped
Result:          More robust, generalizable features
```

**2. Ensemble Effect**
```
Each training step: Different sub-network (due to random drops)
Total training:     Thousands of different networks trained
Inference:          Approximates ensemble of all networks
```

**3. Implicit Regularization**
```
Forces network to learn redundant representations
Similar to L2 regularization but more effective
```

### Where to Apply Dropout in Transformers:

**Standard locations** (BERT, GPT, etc.):

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model)
        self.ffn = FeedForward(d_model)
        
        # Dropout locations:
        self.dropout1 = nn.Dropout(dropout)  # After attention
        self.dropout2 = nn.Dropout(dropout)  # After FFN
        self.dropout3 = nn.Dropout(dropout)  # Inside attention (on weights)
        self.dropout4 = nn.Dropout(dropout)  # Inside FFN (after activation)
    
    def forward(self, x):
        # Attention block
        attn_out = self.attention(x)  # Dropout inside attention
        x = x + self.dropout1(attn_out)  # Dropout after attention
        
        # FFN block
        ffn_out = self.ffn(x)  # Dropout inside FFN
        x = x + self.dropout2(ffn_out)  # Dropout after FFN
        return x
```

**Specific dropout locations**:

1. **Attention Dropout**: On attention weights
   ```python
   attn_weights = F.softmax(scores, dim=-1)
   attn_weights = F.dropout(attn_weights, p=dropout, training=self.training)
   ```

2. **Residual Dropout**: After each sub-layer
   ```python
   x = x + dropout(sublayer(x))
   ```

3. **Embedding Dropout**: On token/position embeddings
   ```python
   embeddings = dropout(token_emb + pos_emb)
   ```

4. **FFN Dropout**: After activation function
   ```python
   x = dropout(F.gelu(linear1(x)))
   ```

### Typical Dropout Values:

```
BERT:              0.1 (all locations)
GPT-2:             0.1 (all locations)
GPT-3:             0.0 (no dropout! trained on huge data)
Llama:             0.0 (no dropout, large pre-training)
T5:                0.1 (all locations)

Fine-tuning:
- Small dataset (<10K):   0.2-0.3
- Medium dataset (10K-100K): 0.1-0.2
- Large dataset (>100K):  0.0-0.1
```

### Dropout Patterns:

**Increasing dropout** (more aggressive):
```python
# Later layers get more dropout
for i, layer in enumerate(layers):
    dropout_p = base_dropout * (1 + i / len(layers))
    layer.dropout = nn.Dropout(dropout_p)
```

**Targeted dropout** (empirical finding):
```python
# FFN dropout higher than attention dropout
attention_dropout = 0.1
ffn_dropout = 0.2
```

In [ ]:
# Demonstrate dropout in transformers
print("=" * 60)
print("DROPOUT IN TRANSFORMERS")
print("=" * 60)

# Simple transformer block with configurable dropout
class SimpleTransformerBlock(nn.Module):
    """Transformer block with dropout at all standard locations."""
    def __init__(self, d_model=256, n_heads=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(
            d_model, 
            n_heads, 
            dropout=dropout,  # Attention dropout
            batch_first=True
        )
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),  # FFN dropout
            nn.Linear(d_model * 4, d_model),
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Residual dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x):
        # Attention block with residual
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # FFN block with residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x

# Test dropout behavior
print("\nDropout Behavior Test:")
print("-" * 60)

torch.manual_seed(42)
batch_size = 2
seq_len = 5
d_model = 64

x = torch.randn(batch_size, seq_len, d_model)
block = SimpleTransformerBlock(d_model=d_model, dropout=0.5)  # High dropout for visibility

# Training mode: Dropout active
block.train()
out_train1 = block(x)
out_train2 = block(x)

print("Training mode (dropout active):")
print(f"  Input:        {x[0, 0, :5]}")
print(f"  Output pass 1: {out_train1[0, 0, :5]}")
print(f"  Output pass 2: {out_train2[0, 0, :5]}")
print(f"  Outputs differ: {not torch.allclose(out_train1, out_train2)}")

# Evaluation mode: No dropout
block.eval()
out_eval1 = block(x)
out_eval2 = block(x)

print("\nEvaluation mode (no dropout):")
print(f"  Output pass 1: {out_eval1[0, 0, :5]}")
print(f"  Output pass 2: {out_eval2[0, 0, :5]}")
print(f"  Outputs identical: {torch.allclose(out_eval1, out_eval2)}")

# Compare different dropout rates
print("\n" + "=" * 60)
print("DROPOUT RATE COMPARISON")
print("=" * 60)

# Create simple classification model
class SimpleTransformerClassifier(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_layers=2, n_classes=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(100, d_model)
        self.dropout_emb = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            SimpleTransformerBlock(d_model, n_heads=4, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        self.classifier = nn.Linear(d_model, n_classes)
    
    def forward(self, x):
        batch_size, seq_len = x.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.dropout_emb(x)
        
        # Transformer layers
        for layer in self.layers:
            x = layer(x)
        
        # Classification (use [CLS] token - first token)
        x = x[:, 0, :]  # [batch_size, d_model]
        logits = self.classifier(x)
        return logits

# Create synthetic dataset
torch.manual_seed(42)
n_train = 500
n_val = 100
seq_len = 20

# Create pattern: class 0 if first few tokens sum > threshold
X_train = torch.randint(0, 100, (n_train, seq_len))
y_train = (X_train[:, :5].float().mean(dim=1) > 50).long()

X_val = torch.randint(0, 100, (n_val, seq_len))
y_val = (X_val[:, :5].float().mean(dim=1) > 50).long()

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Train with different dropout rates
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.5]
results = {}

for dropout in dropout_rates:
    print(f"\nTraining with dropout={dropout}...")
    
    torch.manual_seed(42)
    model = SimpleTransformerClassifier(dropout=dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    val_losses = []
    val_accs = []
    
    for epoch in range(30):
        # Train
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_losses.append(train_loss / len(train_loader))
        
        # Validate
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
                preds = outputs.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
        
        val_losses.append(val_loss / len(val_loader))
        val_accs.append(100 * correct / n_val)
    
    results[dropout] = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs
    }
    
    print(f"  Final train loss: {train_losses[-1]:.4f}")
    print(f"  Final val loss:   {val_losses[-1]:.4f}")
    print(f"  Final val acc:    {val_accs[-1]:.1f}%")

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = plt.cm.viridis(np.linspace(0, 1, len(dropout_rates)))

# Training loss
for i, dropout in enumerate(dropout_rates):
    axes[0].plot(results[dropout]['train_losses'], 
                label=f'dropout={dropout}', color=colors[i], linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss vs Dropout Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation loss
for i, dropout in enumerate(dropout_rates):
    axes[1].plot(results[dropout]['val_losses'], 
                label=f'dropout={dropout}', color=colors[i], linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Validation Loss vs Dropout Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Validation accuracy
for i, dropout in enumerate(dropout_rates):
    axes[2].plot(results[dropout]['val_accs'], 
                label=f'dropout={dropout}', color=colors[i], linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Validation Accuracy (%)')
axes[2].set_title('Validation Accuracy vs Dropout Rate')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY OBSERVATIONS")
print("=" * 60)
print("\n1. No dropout (0.0):")
print("   - Fastest training convergence")
print("   - May overfit on small datasets")
print("   - Good for large pre-training")

print("\n2. Low dropout (0.1):")
print("   - Standard for BERT/GPT fine-tuning")
print("   - Good balance for most tasks")
print("   - Slight regularization")

print("\n3. Medium dropout (0.2-0.3):")
print("   - Better for small datasets")
print("   - Prevents overfitting")
print("   - May slow convergence")

print("\n4. High dropout (0.5+):")
print("   - Very strong regularization")
print("   - May underfit")
print("   - Rarely used in transformers")
print("=" * 60)

## Part 2: Weight Decay - AdamW vs Adam

### The Problem with Adam and L2 Regularization:

**Traditional L2 regularization**:
```python
# Add penalty to loss
loss = criterion(outputs, targets) + lambda * sum(w**2 for w in weights)

# Gradient becomes:
grad = original_grad + lambda * weight
```

**Problem with Adam**:
```
Adam uses adaptive learning rates:
- Divides gradient by running average of gradient magnitudes
- L2 regularization term gets normalized too!
- Result: Weight decay effect is inconsistent
```

**Mathematical Issue**:
```
SGD with L2:
w_new = w_old - lr * (grad + lambda * w_old)
      = w_old * (1 - lr * lambda) - lr * grad
      ← Clear weight decay

Adam with L2:
w_new = w_old - lr * (grad + lambda * w_old) / (sqrt(v) + eps)
      ← L2 term gets divided by adaptive factor!
      ← Weight decay is not consistent!
```

### AdamW: The Solution

**Key insight**: Decouple weight decay from gradient-based update

**Adam** (incorrect weight decay):
```python
# Add L2 to loss
loss = loss + weight_decay * sum(param**2)
grad = grad + weight_decay * param

# Adam update (L2 term gets normalized)
m = beta1 * m + (1 - beta1) * grad
v = beta2 * v + (1 - beta2) * grad**2
param = param - lr * m / (sqrt(v) + eps)
```

**AdamW** (correct weight decay):
```python
# NO L2 in loss
# Adam update (no L2 term)
m = beta1 * m + (1 - beta1) * grad
v = beta2 * v + (1 - beta2) * grad**2
param = param - lr * m / (sqrt(v) + eps)

# THEN apply weight decay directly
param = param * (1 - weight_decay)  # ← Direct decay!
```

### Why AdamW is Better:

**1. Consistent weight decay**:
```
- Weight decay always applied uniformly
- Not affected by gradient magnitude
- More predictable regularization
```

**2. Better hyperparameter tuning**:
```
- Learning rate and weight decay are decoupled
- Can tune them independently
- Easier to find optimal values
```

**3. Empirically better**:
```
- Better generalization on many tasks
- Standard in all modern transformers
- Used in BERT, GPT-2/3, T5, Llama
```

### Typical Weight Decay Values:

```
BERT:          0.01
GPT-2:         0.01
GPT-3:         0.1
T5:            0.01
Llama:         0.1

Fine-tuning:
- Small models (<500M):   0.01-0.05
- Large models (>1B):     0.05-0.1
- With LoRA:              0.01 (or 0.0)
```

### What Parameters to Decay:

**Standard practice**: Decay weights, not biases or LayerNorm

```python
# Separate parameters
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if 'bias' in name or 'norm' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

# Create optimizer with different weight decay
optimizer = optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.01},
    {'params': no_decay_params, 'weight_decay': 0.0}
], lr=1e-4)
```

In [ ]:
# Compare Adam vs AdamW
print("=" * 60)
print("ADAM vs ADAMW COMPARISON")
print("=" * 60)

# Use same model and data from previous section
torch.manual_seed(42)

# Test configurations
configs = [
    {'name': 'Adam (no weight decay)', 'opt': 'Adam', 'wd': 0.0},
    {'name': 'Adam (L2 regularization)', 'opt': 'Adam', 'wd': 0.01},
    {'name': 'AdamW (small decay)', 'opt': 'AdamW', 'wd': 0.01},
    {'name': 'AdamW (medium decay)', 'opt': 'AdamW', 'wd': 0.05},
    {'name': 'AdamW (large decay)', 'opt': 'AdamW', 'wd': 0.1},
]

results = {}

for config in configs:
    print(f"\nTraining: {config['name']}")
    
    torch.manual_seed(42)
    model = SimpleTransformerClassifier(dropout=0.1).to(device)
    
    # Create optimizer
    if config['opt'] == 'Adam':
        optimizer = optim.Adam(
            model.parameters(), 
            lr=0.001, 
            weight_decay=config['wd']  # L2 regularization
        )
    else:  # AdamW
        # Separate parameters (proper way)
        decay_params = []
        no_decay_params = []
        
        for name, param in model.named_parameters():
            if 'bias' in name or 'norm' in name:
                no_decay_params.append(param)
            else:
                decay_params.append(param)
        
        optimizer = optim.AdamW([
            {'params': decay_params, 'weight_decay': config['wd']},
            {'params': no_decay_params, 'weight_decay': 0.0}
        ], lr=0.001)
    
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    val_losses = []
    val_accs = []
    weight_norms = []  # Track weight magnitude
    
    for epoch in range(30):
        # Train
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_losses.append(train_loss / len(train_loader))
        
        # Validate
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
                preds = outputs.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
        
        val_losses.append(val_loss / len(val_loader))
        val_accs.append(100 * correct / n_val)
        
        # Track weight norm
        total_norm = sum(p.data.norm(2).item()**2 for p in model.parameters())**0.5
        weight_norms.append(total_norm)
    
    results[config['name']] = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'weight_norms': weight_norms
    }
    
    print(f"  Final val acc: {val_accs[-1]:.1f}%")
    print(f"  Final weight norm: {weight_norms[-1]:.2f}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = plt.cm.tab10(np.arange(len(configs)))

# Training loss
for i, config in enumerate(configs):
    name = config['name']
    axes[0, 0].plot(results[name]['train_losses'], 
                   label=name, color=colors[i], linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].set_title('Training Loss Comparison')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Validation loss
for i, config in enumerate(configs):
    name = config['name']
    axes[0, 1].plot(results[name]['val_losses'], 
                   label=name, color=colors[i], linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Validation Loss')
axes[0, 1].set_title('Validation Loss Comparison')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Validation accuracy
for i, config in enumerate(configs):
    name = config['name']
    axes[1, 0].plot(results[name]['val_accs'], 
                   label=name, color=colors[i], linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Accuracy (%)')
axes[1, 0].set_title('Validation Accuracy Comparison')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Weight norms
for i, config in enumerate(configs):
    name = config['name']
    axes[1, 1].plot(results[name]['weight_norms'], 
                   label=name, color=colors[i], linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Weight Norm')
axes[1, 1].set_title('Weight Norm Over Training')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY OBSERVATIONS")
print("=" * 60)
print("\n1. AdamW vs Adam:")
print("   - AdamW provides more consistent regularization")
print("   - Better control over weight decay strength")
print("   - Generally better generalization")

print("\n2. Weight decay effect:")
print("   - Keeps weight norms smaller")
print("   - Prevents overfitting")
print("   - Trade-off: may underfit if too high")

print("\n3. Optimal weight decay:")
print("   - 0.01 is standard for BERT/GPT fine-tuning")
print("   - Increase for small datasets")
print("   - Decrease for large datasets or LoRA")
print("=" * 60)

## Part 3: Early Stopping, Gradient Clipping, and Learning Rate Warmup

### Early Stopping

**Idea**: Stop training when validation performance stops improving

**Why it works**:
```
Training curve:
Epoch 1-10:  Val loss decreases ↓
Epoch 11-15: Val loss plateaus →
Epoch 16+:   Val loss increases ↑ (overfitting!)

Early stopping: Stop at epoch 15
```

**Implementation with patience**:
```python
best_val_loss = float('inf')
patience = 5  # Wait 5 epochs before stopping
patience_counter = 0
best_model = None

for epoch in range(max_epochs):
    val_loss = validate()
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        model.load_state_dict(best_model)
        break
```

**Patience values**:
```
Small datasets:   3-5 epochs
Medium datasets:  5-10 epochs
Large datasets:   10-20 epochs
```

### Gradient Clipping

**Problem**: Gradients can explode (become very large)
```
Causes:
- Unstable training
- Loss spikes
- NaN values
- Model divergence
```

**Solution**: Clip gradients to maximum norm

**Two methods**:

**1. Gradient Norm Clipping** (recommended):
```python
# Clip gradient norm to max_norm
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

# What it does:
total_norm = sqrt(sum(grad.norm()**2 for grad in grads))
if total_norm > max_norm:
    scale_factor = max_norm / total_norm
    for grad in grads:
        grad *= scale_factor  # Scale down uniformly
```

**2. Gradient Value Clipping** (less common):
```python
# Clip each gradient value individually
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)
```

**Typical values**:
```
BERT:         1.0
GPT-2/3:      1.0
T5:           1.0
Llama:        1.0

Rule: Start with 1.0, increase if training is too slow
```

### Learning Rate Warmup

**Problem**: Starting with high LR can destabilize training
```
Without warmup:
Epoch 1: Large gradients, large updates → Loss spike!
Result: Training unstable or diverges

With warmup:
Epoch 1: Small LR, small updates → Stable
Gradually increase LR → Full speed
```

**Implementation**:
```python
def get_lr_with_warmup(step, warmup_steps, max_lr):
    if step < warmup_steps:
        # Linear warmup
        return max_lr * (step / warmup_steps)
    else:
        # Constant or decay
        return max_lr

# Usage
warmup_steps = 500
for step in range(total_steps):
    lr = get_lr_with_warmup(step, warmup_steps, max_lr=1e-4)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
```

**Warmup schedules**:
```
BERT:         Linear warmup over 10% of training
GPT-2:        Linear warmup over 2000 steps
T5:           Inverse sqrt: lr = 1/sqrt(max(step, warmup_steps))
Llama:        Cosine with warmup
```

**Typical warmup durations**:
```
Small models:      500-1000 steps
Large models:      2000-5000 steps
Percentage-based:  5-10% of total training steps
```

In [ ]:
# Demonstrate early stopping, gradient clipping, and LR warmup
print("=" * 60)
print("REGULARIZATION TECHNIQUES: COMPREHENSIVE COMPARISON")
print("=" * 60)

# Training function with all regularization techniques
def train_with_regularization(
    model, 
    train_loader, 
    val_loader,
    optimizer,
    criterion,
    max_epochs=50,
    early_stopping_patience=None,
    gradient_clip=None,
    warmup_steps=0,
    max_lr=0.001,
    device='cpu'
):
    """Train with configurable regularization."""
    train_losses = []
    val_losses = []
    val_accs = []
    learning_rates = []
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    stopped_epoch = max_epochs
    
    global_step = 0
    
    for epoch in range(max_epochs):
        # Training
        model.train()
        train_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Learning rate warmup
            if warmup_steps > 0 and global_step < warmup_steps:
                lr = max_lr * (global_step / warmup_steps)
            else:
                lr = max_lr
            
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            
            learning_rates.append(lr)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            if gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            
            optimizer.step()
            
            train_loss += loss.item()
            global_step += 1
        
        train_losses.append(train_loss / len(train_loader))
        
        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
                preds = outputs.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)
        
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        # Early stopping
        if early_stopping_patience is not None:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                stopped_epoch = epoch + 1
                break
    
    # Restore best model if early stopping was used
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'learning_rates': learning_rates,
        'stopped_epoch': stopped_epoch
    }

# Test different regularization combinations
configs = [
    {
        'name': 'No regularization',
        'early_stop': None,
        'grad_clip': None,
        'warmup': 0
    },
    {
        'name': 'Early stopping only',
        'early_stop': 5,
        'grad_clip': None,
        'warmup': 0
    },
    {
        'name': 'Gradient clipping only',
        'early_stop': None,
        'grad_clip': 1.0,
        'warmup': 0
    },
    {
        'name': 'LR warmup only',
        'early_stop': None,
        'grad_clip': None,
        'warmup': 50
    },
    {
        'name': 'All techniques',
        'early_stop': 5,
        'grad_clip': 1.0,
        'warmup': 50
    }
]

all_results = {}

for config in configs:
    print(f"\nTraining: {config['name']}")
    
    torch.manual_seed(42)
    model = SimpleTransformerClassifier(dropout=0.1).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    
    results = train_with_regularization(
        model,
        train_loader,
        val_loader,
        optimizer,
        criterion,
        max_epochs=50,
        early_stopping_patience=config['early_stop'],
        gradient_clip=config['grad_clip'],
        warmup_steps=config['warmup'],
        max_lr=0.001,
        device=device
    )
    
    all_results[config['name']] = results
    
    print(f"  Stopped at epoch: {results['stopped_epoch']}")
    print(f"  Best val loss: {min(results['val_losses']):.4f}")
    print(f"  Best val acc: {max(results['val_accs']):.1f}%")

# Visualize all techniques
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = plt.cm.Set2(np.arange(len(configs)))

# Training loss
for i, config in enumerate(configs):
    name = config['name']
    data = all_results[name]
    axes[0, 0].plot(data['train_losses'], label=name, color=colors[i], linewidth=2)
    if data['stopped_epoch'] < 50:
        axes[0, 0].axvline(data['stopped_epoch'], color=colors[i], linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].set_title('Training Loss Comparison')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Validation loss
for i, config in enumerate(configs):
    name = config['name']
    data = all_results[name]
    axes[0, 1].plot(data['val_losses'], label=name, color=colors[i], linewidth=2)
    if data['stopped_epoch'] < 50:
        axes[0, 1].axvline(data['stopped_epoch'], color=colors[i], linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Validation Loss')
axes[0, 1].set_title('Validation Loss Comparison')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Validation accuracy
for i, config in enumerate(configs):
    name = config['name']
    data = all_results[name]
    axes[1, 0].plot(data['val_accs'], label=name, color=colors[i], linewidth=2)
    if data['stopped_epoch'] < 50:
        axes[1, 0].axvline(data['stopped_epoch'], color=colors[i], linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Accuracy (%)')
axes[1, 0].set_title('Validation Accuracy Comparison')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Learning rate (for warmup comparison)
for i, config in enumerate(configs):
    if config['warmup'] > 0 or config['name'] == 'No regularization':
        name = config['name']
        data = all_results[name]
        # Sample every 10 steps for clarity
        lrs = data['learning_rates'][::10]
        axes[1, 1].plot(lrs, label=name, color=colors[i], linewidth=2)
axes[1, 1].set_xlabel('Training Step (×10)')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary comparison
print("\n" + "=" * 60)
print("SUMMARY COMPARISON")
print("=" * 60)

print(f"\n{'Configuration':<25} {'Best Val Acc':<15} {'Stopped Epoch':<15}")
print("-" * 60)
for config in configs:
    name = config['name']
    data = all_results[name]
    best_acc = max(data['val_accs'])
    stopped = data['stopped_epoch']
    print(f"{name:<25} {best_acc:>6.2f}%         {stopped:>3d}")

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
print("\n1. Early stopping:")
print("   - Prevents overfitting by stopping at optimal point")
print("   - Saves training time")
print("   - Essential for small datasets")

print("\n2. Gradient clipping:")
print("   - Stabilizes training")
print("   - Prevents exploding gradients")
print("   - Allows higher learning rates")

print("\n3. Learning rate warmup:")
print("   - Smooth start to training")
print("   - Prevents early instability")
print("   - Standard in transformer training")

print("\n4. Combined effect:")
print("   - All techniques complement each other")
print("   - Best results with all enabled")
print("   - Standard practice in modern fine-tuning")
print("=" * 60)

## 📊 Summary

### What We Learned:

**1. Dropout in Transformers**:
- Apply at 4 locations: attention weights, after attention, after FFN, embeddings
- Standard rate: 0.1 (BERT/GPT)
- Increase for small datasets (0.2-0.3)
- Prevents co-adaptation, acts like ensemble

**2. Weight Decay - AdamW vs Adam**:
```python
# Adam: L2 gets normalized (wrong)
grad = grad + lambda * weight
update = grad / (sqrt(v) + eps)  # L2 term normalized!

# AdamW: Decoupled weight decay (correct)
update = grad / (sqrt(v) + eps)
weight = weight * (1 - lambda)  # Direct decay!
```
- Always use AdamW for transformers
- Standard: 0.01 for fine-tuning
- Don't decay biases or LayerNorm parameters

**3. Early Stopping**:
```python
patience = 5  # epochs
if val_loss doesn't improve for patience epochs:
    stop and restore best model
```
- Prevents overfitting
- Saves training time
- Patience: 3-5 for small datasets, 10-20 for large

**4. Gradient Clipping**:
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```
- Prevents exploding gradients
- Stabilizes training
- Standard value: 1.0

**5. Learning Rate Warmup**:
```python
if step < warmup_steps:
    lr = max_lr * (step / warmup_steps)
```
- Linear increase from 0 to max_lr
- Prevents early instability
- Warmup: 5-10% of total training steps

### Standard Fine-Tuning Configuration:

```python
# BERT/GPT-style fine-tuning (proven recipe)
config = {
    # Model
    'dropout': 0.1,  # All locations
    
    # Optimizer
    'optimizer': 'AdamW',
    'learning_rate': 2e-5,  # or 3e-5, 5e-5
    'weight_decay': 0.01,
    'betas': (0.9, 0.999),
    'eps': 1e-8,
    
    # Training
    'batch_size': 16,  # or 32
    'epochs': 3,  # or 4, 5
    'warmup_ratio': 0.1,  # 10% warmup
    'gradient_clip': 1.0,
    
    # Early stopping
    'patience': 3,  # epochs
}
```

### When to Adjust:

**Small dataset (<10K samples)**:
```python
'dropout': 0.2-0.3,  # More regularization
'weight_decay': 0.05-0.1,  # Higher decay
'patience': 3,  # Early stopping important
```

**Large dataset (>100K samples)**:
```python
'dropout': 0.0-0.1,  # Less regularization
'weight_decay': 0.01,  # Standard
'patience': 10,  # More patience
```

**Large model (>7B parameters)**:
```python
'learning_rate': 1e-5,  # Lower LR
'weight_decay': 0.1,  # Higher decay
'gradient_clip': 1.0,  # Essential
```

**LoRA fine-tuning**:
```python
'dropout': 0.05,  # LoRA has dropout
'weight_decay': 0.0-0.01,  # Less needed
'learning_rate': 1e-4,  # Can be higher
```

### Implementation Checklist:

**✓ Before training**:
- [ ] Set dropout rate (0.1 default)
- [ ] Use AdamW optimizer
- [ ] Configure weight decay (0.01 default)
- [ ] Exclude biases/norms from weight decay
- [ ] Set up gradient clipping (1.0 default)
- [ ] Configure LR warmup (10% of training)
- [ ] Set up early stopping (patience 3-5)

**✓ During training**:
- [ ] Monitor train vs val loss (gap = overfitting)
- [ ] Check gradient norms (spike = need clipping)
- [ ] Watch for early stopping trigger
- [ ] Verify warmup schedule working

**✓ After training**:
- [ ] Load best model (from early stopping)
- [ ] Evaluate on test set
- [ ] Check train/val/test gap

### Common Mistakes:

**❌ DON'T**:
```python
# Using Adam instead of AdamW
optimizer = torch.optim.Adam(...)  # Wrong!

# Applying weight decay to all parameters
optimizer = AdamW(model.parameters(), weight_decay=0.01)  # Wrong!

# No gradient clipping
optimizer.step()  # May explode!

# No warmup
# Training may be unstable early on

# Training until the end without early stopping
# May overfit
```

**✓ DO**:
```python
# Use AdamW
optimizer = torch.optim.AdamW(...)

# Separate weight decay for different parameters
decay_params = [p for n, p in model.named_parameters() 
                if 'bias' not in n and 'norm' not in n]
no_decay_params = [p for n, p in model.named_parameters() 
                   if 'bias' in n or 'norm' in n]
optimizer = AdamW([
    {'params': decay_params, 'weight_decay': 0.01},
    {'params': no_decay_params, 'weight_decay': 0.0}
])

# Clip gradients
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()

# Use warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps, num_training_steps
)

# Implement early stopping
early_stopper = EarlyStopping(patience=5)
```

### Next Notebook:

**18_evaluation_metrics.ipynb**

We'll explore:
- Perplexity for language models
- BLEU and ROUGE for text generation
- F1, precision, recall for classification
- Exact match for question answering
- Human evaluation strategies
- Creating comprehensive evaluation reports

---

**Historical Note**: The combination of these regularization techniques (dropout, AdamW, clipping, warmup, early stopping) became the "standard recipe" around 2018-2019 with BERT and GPT-2. Before this, each technique was used in isolation. The insight that they work best together, and that specific values (dropout=0.1, weight_decay=0.01, clip=1.0) work across many tasks, revolutionized fine-tuning practice. This recipe now underlies almost all successful transformer fine-tuning.